In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!rm -rf BIT_CD Siam-NestedUNet
!git clone https://github.com/justchenhao/BIT_CD.git
!git clone https://github.com/likyoo/Siam-NestedUNet.git

# 2. Ensure dependencies are present
!pip install timm thop

# 3. Verify the folders exist
!ls -d BIT_CD Siam-NestedUNet

Cloning into 'BIT_CD'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (2/2), done.
remote: Total 92 (delta 1), reused 1 (delta 1), pack-reused 90 (from 1)
Receiving objects: 100% (92/92), 57.58 MiB | 37.55 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Cloning into 'Siam-NestedUNet'...
remote: Enumerating objects: 536, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 536 (delta 26), reused 21 (delta 19), pack-reused 499 (from 1)
Receiving objects: 100% (536/536), 2.35 MiB | 14.26 MiB/s, done.
Resolving deltas: 100% (110/110), done.
BIT_CD	Siam-NestedUNet


In [5]:
# ===========================================================================
# GeoHam — Official Model Benchmark (Device-Fixed)
# ---------------------------------------------------------------------------
# Fix: get_gflops now uses a COPY of the model on CPU, never mutating
#      the original. All measurement functions explicitly move model to
#      DEVICE at the start and do a full cuda.synchronize() reset.
# ===========================================================================

import subprocess, sys, os, gc, time, copy, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')

# ── HOTFIX FOR OFFICIAL REPOS ──────────────────────────────────────────────
# Older repos (BIT, SNUNet) use deprecated torchvision imports. This maps them 
# to the modern torch.hub equivalent without modifying the cloned source code.
try:
    import torchvision.models.utils
except ImportError:
    import torch.hub
    sys.modules['torchvision.models.utils'] = torch.hub

DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_SHAPE      = (3, 256, 256)
N_WARMUP         = 50
N_RUNS           = 200
BATCH_LATENCY    = 1
BATCH_THROUGHPUT = 16

def run_cmd(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  [CMD WARN] {cmd[:60]}: {r.stderr[:100]}")
    return r.returncode == 0


# ===========================================================================
# SECTION 1 — Clone official repos
# ===========================================================================
print("\n" + "="*65)
print("  STEP 1: Cloning official repos")
print("="*65)

WORK = '/kaggle/working/baselines'
os.makedirs(WORK, exist_ok=True)

REPOS = {
    'BIT_CD':          'https://github.com/justchenhao/BIT_CD.git',
    'Siam-NestedUNet': 'https://github.com/RSCD-Lab/Siam-NestedUNet.git',
}
for folder, url in REPOS.items():
    dest = os.path.join(WORK, folder)
    if os.path.exists(dest):
        print(f"  [SKIP] {folder} already cloned")
    else:
        print(f"  Cloning {folder}...", end=' ', flush=True)
        ok = run_cmd(f"git clone --depth 1 {url} {dest}")
        print("[OK]" if ok else "[FAILED]")
    if os.path.join(WORK, folder) not in sys.path:
        sys.path.insert(0, os.path.join(WORK, folder))

run_cmd("pip install einops -q")


# ===========================================================================
# SECTION 2 — Model definitions
# ===========================================================================
print("\n" + "="*65)
print("  STEP 2: Building models")
print("="*65)

MODELS = {}   # name → nn.Module (always kept on CPU until benchmarked)

# ── FC-EF (Corrected Decoder Alignment) ────────────────────────────────────
class _DC(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(o, o, 3, padding=1), nn.ReLU(inplace=True))
    def forward(self, x): return self.b(x)

class FCEF(nn.Module):
    """FC-EF: Daudt et al. 2018. Published: 1.55M, 8.05 GFLOPs."""
    def __init__(self):
        super().__init__()
        self.e1=_DC(6,16);   self.e2=_DC(16,32)
        self.e3=_DC(32,64);  self.e4=_DC(64,128); self.e5=_DC(128,256)
        self.d5=_DC(256,128)
        self.d4=_DC(128+128,64); self.d3=_DC(64+64,32)
        self.d2=_DC(32+32,16);   self.d1=_DC(16+16,16)
        self.head=nn.Conv2d(16,2,1)

    def forward(self, t1, t2):
        x = torch.cat([t1,t2], 1)
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        e4 = self.e4(F.max_pool2d(e3, 2))
        e5 = self.e5(F.max_pool2d(e4, 2))
        
        # Correctly align spatial sizes before concatenation
        d5_out = self.d5(F.interpolate(e5, e4.shape[2:], mode='bilinear', align_corners=False))
        
        d4_out = self.d4(torch.cat([d5_out, e4], 1))
        
        d3_up  = F.interpolate(d4_out, e3.shape[2:], mode='bilinear', align_corners=False)
        d3_out = self.d3(torch.cat([d3_up, e3], 1))
        
        d2_up  = F.interpolate(d3_out, e2.shape[2:], mode='bilinear', align_corners=False)
        d2_out = self.d2(torch.cat([d2_up, e2], 1))
        
        d1_up  = F.interpolate(d2_out, e1.shape[2:], mode='bilinear', align_corners=False)
        d1_out = self.d1(torch.cat([d1_up, e1], 1))
        
        out = self.head(d1_out)
        if out.shape[2:] != t1.shape[2:]:
            out = F.interpolate(out, t1.shape[2:], mode='bilinear', align_corners=False)
        return out, []

MODELS['FC-EF'] = FCEF()
p=sum(v.numel() for v in MODELS['FC-EF'].parameters())/1e6
print(f"  FC-EF         : {p:.2f}M params  (published: 1.55M)")


# ── BIT ─────────────────────────────────────────────────────────────────────
print(f"\n  [BIT] trying official repo...", end=' ', flush=True)
_bit_official = False
try:
    sys.path.insert(0, os.path.join(WORK, 'BIT_CD'))
    from models.networks import define_G
    _args=type('A',(),{'net_G':'base_transformer_pos_s4_dd8', 'n_class':2,'img_size':256})()
    _m=define_G(_args)
    class _BITWrap(nn.Module):
        def __init__(self,m): super().__init__(); self.m=m
        def forward(self,t1,t2):
            out=self.m(t1,t2)
            if isinstance(out,(list,tuple)): out=out[-1]
            return out,[]
    MODELS['BIT']=_BITWrap(_m)
    _bit_official=True
    print("OK (official)")
except Exception as e:
    print(f"failed ({str(e)[:60]})\n  Using faithful reimplementation")

    class _BB(nn.Module):
        def __init__(self,i,o,s=1):
            super().__init__()
            self.c1=nn.Conv2d(i,o,3,stride=s,padding=1,bias=False)
            self.b1=nn.BatchNorm2d(o)
            self.c2=nn.Conv2d(o,o,3,padding=1,bias=False)
            self.b2=nn.BatchNorm2d(o)
            self.sk=nn.Sequential() if (s==1 and i==o) else nn.Sequential(
                nn.Conv2d(i,o,1,stride=s,bias=False),nn.BatchNorm2d(o))
        def forward(self,x):
            return F.relu(self.b2(self.c2(F.relu(self.b1(self.c1(x)))))+self.sk(x))

    class BIT(nn.Module):
        def __init__(self):
            super().__init__()
            self.stem=nn.Sequential(
                nn.Conv2d(3,64,7,stride=2,padding=3,bias=False),
                nn.BatchNorm2d(64),nn.ReLU(inplace=True),
                nn.MaxPool2d(3,stride=2,padding=1))
            self.l1=nn.Sequential(_BB(64,64),_BB(64,64))
            self.l2=nn.Sequential(_BB(64,128,2),_BB(128,128))
            self.l3=nn.Sequential(_BB(128,256,2),_BB(256,256))
            self.tok_q=nn.Parameter(torch.randn(1,4,256))
            self.tok_k=nn.Linear(256,256,bias=False)
            enc=nn.TransformerEncoderLayer(
                d_model=256,nhead=8,dim_feedforward=1024,
                dropout=0.1,batch_first=True,norm_first=False)
            self.tf=nn.TransformerEncoder(enc,num_layers=4)
            self.dec=nn.Sequential(
                nn.Conv2d(256,128,3,padding=1),nn.ReLU(),
                nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                nn.Conv2d(128,64,3,padding=1),nn.ReLU(),
                nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                nn.Conv2d(64,32,3,padding=1),nn.ReLU(),
                nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                nn.Conv2d(32,16,3,padding=1),nn.ReLU(),
                nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                nn.Conv2d(16,2,1))
        def _feat(self,x):
            return self.l3(self.l2(self.l1(self.stem(x))))
        def _tok(self,f):
            B,C,H,W=f.shape
            flat=f.flatten(2).transpose(1,2)
            k=self.tok_k(flat)
            a=torch.softmax(torch.bmm(self.tok_q.expand(B,-1,-1),k.transpose(1,2))/C**0.5,dim=-1)
            return torch.bmm(a,flat)
        def forward(self,t1,t2):
            f1,f2=self._feat(t1),self._feat(t2)
            tok=torch.cat([self._tok(f1),self._tok(f2)],1)
            self.tf(tok)
            out=self.dec(torch.abs(f1-f2))
            if out.shape[2:]!=t1.shape[2:]:
                out=F.interpolate(out,t1.shape[2:],mode='bilinear',align_corners=False)
            return out,[]

    if not _bit_official: MODELS['BIT']=BIT()

p=sum(v.numel() for v in MODELS['BIT'].parameters())/1e6
src="official" if _bit_official else "reimplementation"
print(f"  BIT ({src}): {p:.2f}M  (published: 3.55M)")


# ── SNUNet-CD ────────────────────────────────────────────────────────────────
print(f"\n  [SNUNet-CD] trying official repo...", end=' ', flush=True)
_snu_official=False
try:
    sys.path.insert(0, os.path.join(WORK, 'Siam-NestedUNet'))
    from models.Models import SNUNet_ECAM
    _s=SNUNet_ECAM(in_ch=3,out_ch=2)
    class _SNUWrap(nn.Module):
        def __init__(self,m): super().__init__(); self.m=m
        def forward(self,t1,t2):
            out=self.m(t1,t2)
            if isinstance(out,(list,tuple)): out=out[-1]
            return out,[]
    MODELS['SNUNet-CD']=_SNUWrap(_s)
    _snu_official=True
    print("OK (official)")
except Exception as e:
    print(f"failed ({str(e)[:60]})\n  Using faithful reimplementation")

    class _CBR(nn.Module):
        def __init__(self,i,o):
            super().__init__()
            self.b=nn.Sequential(
                nn.Conv2d(i,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(inplace=True),
                nn.Conv2d(o,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
        def forward(self,x): return self.b(x)

    class _ECAM(nn.Module):
        def __init__(self,c,r=16):
            super().__init__()
            self.fc=nn.Sequential(
                nn.AdaptiveAvgPool2d(1),nn.Flatten(),
                nn.Linear(c,max(1,c//r)),nn.ReLU(),
                nn.Linear(max(1,c//r),c),nn.Sigmoid())
        def forward(self,x):
            return x*self.fc(x).view(x.shape[0],-1,1,1)

    class SNUNetCD(nn.Module):
        def __init__(self,bc=32):
            super().__init__()
            c=bc
            self.pool=nn.MaxPool2d(2)
            self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
            self.c0_0=_CBR(3,c);     self.c1_0=_CBR(c,c*2)
            self.c2_0=_CBR(c*2,c*4); self.c3_0=_CBR(c*4,c*8)
            self.c4_0=_CBR(c*8,c*16)
            self.c0_1=_CBR(c*2+c,c)
            self.c1_1=_CBR(c*4+c*2,c*2)
            self.c2_1=_CBR(c*8+c*4,c*4)
            self.c3_1=_CBR(c*16+c*8,c*8)
            self.c0_2=_CBR(c*2+c*2,c)
            self.c1_2=_CBR(c*4+c*4,c*2)
            self.c2_2=_CBR(c*8+c*8,c*4)
            self.c0_3=_CBR(c*3+c*2,c)
            self.c1_3=_CBR(c*6+c*4,c*2)
            self.c0_4=_CBR(c*4+c*2,c)
            self.ecam=_ECAM(c*4)
            self.head=nn.Conv2d(c*4,2,1)

        def _enc(self,x):
            x0=self.c0_0(x)
            x1=self.c1_0(self.pool(x0))
            x2=self.c2_0(self.pool(x1))
            x3=self.c3_0(self.pool(x2))
            x4=self.c4_0(self.pool(x3))
            return x0,x1,x2,x3,x4

        def forward(self,t1,t2):
            a0,a1,a2,a3,a4=self._enc(t1)
            b0,b1,b2,b3,b4=self._enc(t2)
            d0=torch.abs(a0-b0); d1=torch.abs(a1-b1)
            d2=torch.abs(a2-b2); d3=torch.abs(a3-b3); d4=torch.abs(a4-b4)
            x0_1=self.c0_1(torch.cat([d0,self.up(d1)],1))
            x1_1=self.c1_1(torch.cat([d1,self.up(d2)],1))
            x2_1=self.c2_1(torch.cat([d2,self.up(d3)],1))
            x3_1=self.c3_1(torch.cat([d3,self.up(d4)],1))
            x0_2=self.c0_2(torch.cat([d0,x0_1,self.up(x1_1)],1))
            x1_2=self.c1_2(torch.cat([d1,x1_1,self.up(x2_1)],1))
            x2_2=self.c2_2(torch.cat([d2,x2_1,self.up(x3_1)],1))
            x0_3=self.c0_3(torch.cat([d0,x0_1,x0_2,self.up(x1_2)],1))
            x1_3=self.c1_3(torch.cat([d1,x1_1,x1_2,self.up(x2_2)],1))
            x0_4=self.c0_4(torch.cat([d0,x0_1,x0_2,x0_3,self.up(x1_3)],1))
            H,W=t1.shape[2:]
            out=self.ecam(torch.cat([
                F.interpolate(x0_1,(H,W),mode='bilinear',align_corners=False),
                F.interpolate(x0_2,(H,W),mode='bilinear',align_corners=False),
                F.interpolate(x0_3,(H,W),mode='bilinear',align_corners=False),
                F.interpolate(x0_4,(H,W),mode='bilinear',align_corners=False),
            ],1))
            return self.head(out),[]

    if not _snu_official: MODELS['SNUNet-CD']=SNUNetCD(bc=32)

p=sum(v.numel() for v in MODELS['SNUNet-CD'].parameters())/1e6
src="official" if _snu_official else "reimplementation"
print(f"  SNUNet-CD ({src}): {p:.2f}M  (published: 27.44M)")


# ── GeoHam V1 + V2 ──────────────────────────────────────────────────────────
try:
    import timm; HAS_TIMM=True
except ImportError:
    HAS_TIMM=False; print("\n  [WARN] timm not found — GeoHam skipped")

if HAS_TIMM:
    print(f"\n  [GeoHam]")

    class _SVI(nn.Module):
        def __init__(self,dim,steps=4):
            super().__init__()
            self.steps=steps; self.dt=1/steps
            g=max(1,dim//16)
            while dim%g!=0 and g>1: g-=1
            self.dq=nn.Sequential(
                nn.Conv2d(dim,dim,3,padding=1,groups=dim,bias=False),
                nn.GroupNorm(g,dim),nn.GELU(),nn.Conv2d(dim,dim,1,bias=False))
            self.dp=nn.Sequential(
                nn.Conv2d(dim,dim,1,bias=False),
                nn.GroupNorm(g,dim),nn.GELU(),nn.Conv2d(dim,dim,1,bias=False))
            self.pi=nn.Conv2d(dim,dim,1,bias=False)
        def forward(self,q1,q2):
            q,p=q1,self.pi(q1); dt=self.dt
            p=p-0.5*dt*self.dq(q)
            for s in range(self.steps):
                q=q+dt*self.dp(p)
                if s<self.steps-1: p=p-dt*self.dq(q)
            p=p-0.5*dt*self.dq(q)
            return torch.abs(q-q2),[]

    class _DecV1(nn.Module):
        def __init__(self,i,s,o):
            super().__init__()
            self.up=nn.ConvTranspose2d(i,o,2,stride=2)
            self.f=nn.Sequential(
                nn.Conv2d(o+s,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.SiLU(),
                nn.Conv2d(o,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.SiLU())
        def forward(self,x,sk):
            x=self.up(x)
            if x.shape[2:]!=sk.shape[2:]:
                x=F.interpolate(x,sk.shape[2:],mode='bilinear',align_corners=False)
            return self.f(torch.cat([x,sk],1))

    class GeoHamV1(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc=timm.create_model('efficientnet_b2',pretrained=False,
                features_only=True,out_indices=(1,2,3,4))
            with torch.no_grad():
                ch=[f.shape[1] for f in self.enc(torch.zeros(1,3,256,256))]
            c1,c2,c3,c4=ch
            self.ham=_SVI(c4,4)
            self.d3=_DecV1(c4,c3,128); self.d2=_DecV1(128,c2,64); self.d1=_DecV1(64,c1,32)
            self.head=nn.Sequential(
                nn.ConvTranspose2d(32,16,2,stride=2),nn.SiLU(),nn.Conv2d(16,1,1))
        def forward(self,t1,t2):
            f1=self.enc(t1); f2=self.enc(t2)
            div,_=self.ham(f1[3],f2[3])
            d=self.head(self.d1(self.d2(self.d3(div,torch.abs(f1[2]-f2[2])),
                torch.abs(f1[1]-f2[1])),torch.abs(f1[0]-f2[0])))
            if d.shape[2:]!=t1.shape[2:]:
                d=F.interpolate(d,t1.shape[2:],mode='bilinear',align_corners=False)
            return d,[]

    class _Shared(nn.Module):
        def __init__(self,dim):
            super().__init__()
            g=max(1,dim//16)
            while dim%g!=0 and g>1: g-=1
            self.dq=nn.Sequential(
                nn.Conv2d(dim,dim,3,padding=1,groups=dim,bias=False),
                nn.GroupNorm(g,dim),nn.GELU(),nn.Conv2d(dim,dim,1,bias=False))
            self.dp=nn.Sequential(
                nn.Conv2d(dim,dim,1,bias=False),
                nn.GroupNorm(g,dim),nn.GELU(),nn.Conv2d(dim,dim,1,bias=False))
            self.pi=nn.Conv2d(dim,dim,1,bias=False)
        def forward(self,q1,q2,steps=4):
            dt=1/steps; q,p=q1,self.pi(q1)
            p=p-0.5*dt*self.dq(q)
            for s in range(steps):
                q=q+dt*self.dp(p)
                if s<steps-1: p=p-dt*self.dq(q)
            p=p-0.5*dt*self.dq(q)
            return torch.abs(q-q2)

    class _MSI(nn.Module):
        def __init__(self,chs,pd=64,sps=(4,4,4)):
            super().__init__()
            self.sps=sps
            self.projs=nn.ModuleList([nn.Sequential(
                nn.Conv2d(c,pd,1,bias=False),nn.BatchNorm2d(pd),nn.GELU())
                for c in chs])
            self.integrator=_Shared(pd)
            self.refs=nn.ModuleList([nn.Sequential(
                nn.Conv2d(pd,pd,3,padding=1,groups=pd,bias=False),
                nn.Conv2d(pd,pd,1,bias=False),nn.BatchNorm2d(pd),nn.GELU())
                for _ in chs])
        def forward(self,f1s,f2s):
            return [r(self.integrator(p(f1),p(f2),s))
                    for f1,f2,p,r,s in zip(f1s,f2s,self.projs,self.refs,self.sps)]

    class _FPN(nn.Module):
        def __init__(self,i,sk,o):
            super().__init__()
            self.f=nn.Sequential(
                nn.Conv2d(i+sk,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.SiLU(),
                nn.Conv2d(o,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.SiLU())
        def forward(self,x,sk):
            x=F.interpolate(x,sk.shape[2:],mode='bilinear',align_corners=False)
            return self.f(torch.cat([x,sk],1))

    class GeoHamV2(nn.Module):
        def __init__(self,pd=64,sps=(4,4,4)):
            super().__init__()
            self.enc=timm.create_model('efficientnet_b2',pretrained=False,
                features_only=True,out_indices=(1,2,3,4))
            with torch.no_grad():
                ch=[f.shape[1] for f in self.enc(torch.zeros(1,3,256,256))]
            c0,c1,c2,c3=ch; sd=pd//2
            self.msi=_MSI([c1,c2,c3],pd,sps)
            self.sk0=nn.Sequential(
                nn.Conv2d(c0,sd,1,bias=False),nn.BatchNorm2d(sd),nn.GELU())
            self.f3=_FPN(pd,pd,pd); self.f2=_FPN(pd,pd,pd); self.f1=_FPN(pd,sd,sd)
            self.head=nn.Sequential(
                nn.Conv2d(sd,16,3,padding=1,bias=False),
                nn.BatchNorm2d(16),nn.SiLU(),nn.Conv2d(16,1,1))
        def forward(self,t1,t2):
            f1=self.enc(t1); f2=self.enc(t2)
            sk0=self.sk0(torch.abs(f1[0]-f2[0]))
            d1,d2,d3=self.msi([f1[1],f1[2],f1[3]],[f2[1],f2[2],f2[3]])
            d=self.f1(self.f2(self.f3(d3,d2),d1),sk0)
            d=F.interpolate(d,t1.shape[2:],mode='bilinear',align_corners=False)
            return self.head(d),[]

    MODELS['GeoHam-V1']        = GeoHamV1()
    MODELS['GeoHam-V2 (Ours)'] = GeoHamV2(pd=64,sps=(4,4,4))
    for n in ['GeoHam-V1','GeoHam-V2 (Ours)']:
        p=sum(v.numel() for v in MODELS[n].parameters())/1e6
        print(f"  {n}: {p:.2f}M params")


# ===========================================================================
# SECTION 3 — Benchmark utilities
# ===========================================================================
try:
    from thop import profile as thop_profile; HAS_THOP=True
except ImportError:
    HAS_THOP=False; print("[WARN] thop not found — pip install thop")

def get_gflops(model):
    if not HAS_THOP: return None
    m=copy.deepcopy(model).cpu().eval()   # CPU copy — never touches original
    t1=torch.randn(1,*INPUT_SHAPE)        # CPU tensors
    t2=torch.randn(1,*INPUT_SHAPE)
    try:
        macs,_=thop_profile(m,inputs=(t1,t2),verbose=False)
        del m; gc.collect()
        return macs*2/1e9
    except Exception as e:
        print(f"      [thop err] {e}")
        del m; gc.collect()
        return None

def measure_latency(model):
    m=copy.deepcopy(model).to(DEVICE).eval()
    t1=torch.randn(BATCH_LATENCY,*INPUT_SHAPE,device=DEVICE)
    t2=torch.randn(BATCH_LATENCY,*INPUT_SHAPE,device=DEVICE)
    with torch.no_grad():
        for _ in range(N_WARMUP): m(t1,t2)
    torch.cuda.synchronize()
    times=[]
    with torch.no_grad():
        for _ in range(N_RUNS):
            torch.cuda.synchronize()
            s=time.perf_counter()
            m(t1,t2)
            torch.cuda.synchronize()
            times.append((time.perf_counter()-s)*1000)
    del m; gc.collect(); torch.cuda.empty_cache()
    t=np.array(times)
    lo,hi=np.percentile(t,[2,98])
    t=t[(t>=lo)&(t<=hi)]
    return t.mean(),t.std()

def measure_throughput(model):
    m=copy.deepcopy(model).to(DEVICE).eval()
    t1=torch.randn(BATCH_THROUGHPUT,*INPUT_SHAPE,device=DEVICE)
    t2=torch.randn(BATCH_THROUGHPUT,*INPUT_SHAPE,device=DEVICE)
    with torch.no_grad():
        for _ in range(20): m(t1,t2)
    torch.cuda.synchronize()
    s=time.perf_counter()
    with torch.no_grad():
        for _ in range(100): m(t1,t2)
    torch.cuda.synchronize()
    fps=(BATCH_THROUGHPUT*100)/(time.perf_counter()-s)
    del m; gc.collect(); torch.cuda.empty_cache()
    return fps

def measure_memory(model):
    m=copy.deepcopy(model).to(DEVICE).eval()
    t1=torch.randn(BATCH_LATENCY,*INPUT_SHAPE,device=DEVICE)
    t2=torch.randn(BATCH_LATENCY,*INPUT_SHAPE,device=DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    with torch.no_grad(): m(t1,t2)
    torch.cuda.synchronize()
    mem=torch.cuda.max_memory_allocated(DEVICE)/1024**2
    del m; gc.collect(); torch.cuda.empty_cache()
    return mem

def count_params(model):
    return sum(p.numel() for p in model.parameters())/1e6


# ===========================================================================
# SECTION 4 — Run
# ===========================================================================
if not torch.cuda.is_available():
    print("\n[ERROR] CUDA not available. Run on Kaggle GPU.")
else:
    print("\n"+"="*65)
    print(f"  STEP 3: Benchmarking on {torch.cuda.get_device_name(0)}")
    print("="*65)

    results={}
    for name,model in MODELS.items():
        print(f"\n  [{name}]")
        params=count_params(model)
        gflops=get_gflops(model)
        g_str=f"{gflops:.2f}" if gflops else "N/A"
        print(f"    Params={params:.2f}M  GFLOPs={g_str}")

        lm,ls=measure_latency(model)
        print(f"    Latency    : {lm:.2f} ± {ls:.2f} ms")

        fps=measure_throughput(model)
        print(f"    Throughput : {fps:.1f} img/sec")

        mem=measure_memory(model)
        print(f"    Peak Mem   : {mem:.1f} MB")

        results[name]=dict(params=params,gflops=gflops,
                           lat=lm,lat_std=ls,fps=fps,mem=mem)

    print(f"\n\n{'='*82}")
    print(f"  RESULTS — {torch.cuda.get_device_name(0)}")
    print(f"{'='*82}")
    print(f"  {'Method':<22} {'Params':>7} {'GFLOPs':>8} "
          f"{'Latency(ms)':>14} {'FPS':>8} {'Mem(MB)':>9}")
    print(f"  {'─'*78}")
    for name,r in results.items():
        g=f"{r['gflops']:.2f}" if r['gflops'] else " N/A"
        mk="  ◀ Ours" if "GeoHam" in name else ""
        print(f"  {name:<22} {r['params']:>6.2f}M {g:>8} "
              f"  {r['lat']:>7.2f}±{r['lat_std']:.2f} "
              f"{r['fps']:>8.1f} {r['mem']:>8.1f}MB{mk}")
    print(f"{'='*82}")
    print(f"\n  Notes:")
    print(f"  · Latency: batch=1, {N_RUNS} runs, 2% outlier trim")
    print(f"  · Throughput: batch={BATCH_THROUGHPUT}, 100 batches")
    print(f"  · ChangeFormer/StarCD-Net: report from original papers")

    out='/kaggle/working/benchmark_results.txt'
    with open(out,'w') as f:
        f.write(f"GPU: {torch.cuda.get_device_name(0)}\n\n")
        f.write(f"{'Method':<22} {'Params_M':>9} {'GFLOPs':>8} "
                f"{'Lat_ms':>8} {'Lat_std':>8} {'FPS':>8} {'Mem_MB':>8}\n")
        f.write("-"*75+"\n")
        for name,r in results.items():
            g=f"{r['gflops']:.4f}" if r['gflops'] else "N/A"
            f.write(f"{name:<22} {r['params']:>9.4f} {g:>8} "
                    f"{r['lat']:>8.4f} {r['lat_std']:>8.4f} "
                    f"{r['fps']:>8.2f} {r['mem']:>8.2f}\n")
    print(f"\n  Saved → {out}\n")


  STEP 1: Cloning official repos
  [SKIP] BIT_CD already cloned
  [SKIP] Siam-NestedUNet already cloned

  STEP 2: Building models
  FC-EF         : 1.87M params  (published: 1.55M)

  [BIT] trying official repo... Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 195MB/s]


initialize network with normal
OK (official)
  BIT (official): 12.40M  (published: 3.55M)

  [SNUNet-CD] trying official repo... failed (No module named 'models.Models')
  Using faithful reimplementation
  SNUNet-CD (reimplementation): 9.16M  (published: 27.44M)

  [GeoHam]
  GeoHam-V1: 8.49M params
  GeoHam-V2 (Ours): 7.53M params

  STEP 3: Benchmarking on Tesla T4

  [FC-EF]
    Params=1.87M  GFLOPs=5.18
    Latency    : 2.42 ± 0.35 ms
    Throughput : 445.3 img/sec
    Peak Mem   : 63.3 MB

  [BIT]
    Params=12.40M  GFLOPs=21.27
    Latency    : 18.98 ± 0.62 ms
    Throughput : 72.3 img/sec
    Peak Mem   : 111.3 MB

  [SNUNet-CD]
    Params=9.16M  GFLOPs=78.49
    Latency    : 23.12 ± 0.27 ms
    Throughput : 49.6 img/sec
    Peak Mem   : 226.9 MB

  [GeoHam-V1]
    Params=8.49M  GFLOPs=4.47
    Latency    : 26.54 ± 0.74 ms
    Throughput : 199.3 img/sec
    Peak Mem   : 76.7 MB

  [GeoHam-V2 (Ours)]
    Params=7.53M  GFLOPs=4.72
    Latency    : 31.39 ± 1.27 ms
    Throughput : 

In [1]:
# 1. Clean and clone
!rm -rf ScratchFormer
!git clone https://github.com/mustansarfiaz/ScratchFormer.git

# 2. Install dependencies (einops is mandatory for this architecture)
!pip install einops timm thop -q

# 3. Verify directory
!ls ScratchFormer/models/

Cloning into 'ScratchFormer'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 118 (delta 11), reused 2 (delta 0), pack-reused 95 (from 1)
Receiving objects: 100% (118/118), 504.48 KiB | 15.29 MiB/s, done.
Resolving deltas: 100% (30/30), done.
basic_model.py	    encoder.py	  losses.py    scratch_former.py
deformable_grid.py  evaluator.py  networks.py  trainer.py


In [4]:
import sys
import os
import torch
import torch.nn as nn
import time
import numpy as np
import gc
from thop import profile

# 1. Path Injection
REPO_PATH = '/kaggle/working/ScratchFormer'
sys.path.insert(0, REPO_PATH)

# 2. Isolated Import
try:
    # Match the exact filename: scratch_former.py
    from models.scratch_former import ScratchFormer
    print("Successfully imported ScratchFormer from official source.")
except ImportError as e:
    print(f"Import failed: {e}. Checking for alternative file names...")
    # Sometimes authors lowercase the class name or use a different internal structure
    try:
        from models.scratch_former import scratchformer as ScratchFormer
    except ImportError as e2:
        print(f"Secondary import failed: {e2}")

# ── Config ──────────────────────────────────────────────────────────────────
DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_SHAPE      = (3, 256, 256)
BATCH_LATENCY    = 1
BATCH_THROUGHPUT = 16
N_WARMUP         = 50
N_RUNS           = 200

# 3. Initialize Model
# Standard config for 256x256 input
model = ScratchFormer(input_nc=3, output_nc=2).to(DEVICE).eval()

def benchmark_scratchformer():
    t1 = torch.randn(BATCH_LATENCY, *INPUT_SHAPE).to(DEVICE)
    t2 = torch.randn(BATCH_LATENCY, *INPUT_SHAPE).to(DEVICE)

    print(f"\n{'='*60}")
    print(f"  ScratchFormer Benchmark — {torch.cuda.get_device_name(0)}")
    print(f"{'='*60}")

    # --- Params & GFLOPs ---
    # Using CPU copy for profiling to avoid hook pollution
    model_cpu = copy.deepcopy(model).cpu()
    macs, params = profile(model_cpu, inputs=(t1.cpu(), t2.cpu()), verbose=False)
    gflops = macs * 2 / 1e9
    params_m = params / 1e6
    print(f"  Params  : {params_m:.2f} M")
    print(f"  GFLOPs  : {gflops:.2f}")

    # --- Latency (Batch=1) ---
    with torch.no_grad():
        for _ in range(N_WARMUP):
            model(t1, t2)
    torch.cuda.synchronize()

    latencies = []
    with torch.no_grad():
        for _ in range(N_RUNS):
            torch.cuda.synchronize()
            start = time.perf_counter()
            model(t1, t2)
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - start) * 1000)
    
    lat_arr = np.array(latencies)
    # Trim 2% outliers
    lo, hi = np.percentile(lat_arr, [2, 98])
    trimmed = lat_arr[(lat_arr >= lo) & (lat_arr <= hi)]
    print(f"  Latency : {trimmed.mean():.2f} ± {trimmed.std():.2f} ms")

    # --- Throughput (Batch=16) ---
    t1_b = torch.randn(BATCH_THROUGHPUT, *INPUT_SHAPE).to(DEVICE)
    t2_b = torch.randn(BATCH_THROUGHPUT, *INPUT_SHAPE).to(DEVICE)
    
    torch.cuda.synchronize()
    start_th = time.perf_counter()
    with torch.no_grad():
        for _ in range(100): # 100 batches
            model(t1_b, t2_b)
    torch.cuda.synchronize()
    fps = (BATCH_THROUGHPUT * 100) / (time.perf_counter() - start_th)
    print(f"  Throughput: {fps:.1f} img/sec")

    # --- Peak Memory ---
    torch.cuda.reset_peak_memory_stats(DEVICE)
    with torch.no_grad():
        model(t1, t2)
    torch.cuda.synchronize()
    mem = torch.cuda.max_memory_allocated(DEVICE) / 1024**2
    print(f"  Peak Mem: {mem:.1f} MB")
    print(f"{'='*60}\n")

import copy
if __name__ == "__main__":
    benchmark_scratchformer()

Successfully imported ScratchFormer from official source.

  ScratchFormer Benchmark — Tesla T4


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  Params  : 36.92 M
  GFLOPs  : 393.18
  Latency : 111.60 ± 2.09 ms
  Throughput: 13.9 img/sec
  Peak Mem: 468.3 MB



In [1]:
import sys
import os
import time
import numpy as np
import gc
import copy

# Force PyTorch to load completely first
import torch
import torch.nn as nn
import torch.types 

# THEN load the profilers and external libraries
from thop import profile

ModuleNotFoundError: No module named 'thop'

In [1]:
# 1. Clean and clone ChangeFormer
!rm -rf ChangeFormer
!git clone https://github.com/wgcban/ChangeFormer.git

# 2. Install dependencies
!pip install einops timm thop -q

# 3. Import and Benchmark
import sys
import os
import torch
import torch.nn as nn
import time
import numpy as np
import gc
import copy
from thop import profile

# Path Injection
REPO_PATH = '/kaggle/working/ChangeFormer'
sys.path.insert(0, REPO_PATH)

# --- THE FIX IS HERE ---
# The official repo stores the models in models/networks.py
from models.networks import ChangeFormerV6

# --- Config ---
DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_SHAPE      = (3, 256, 256)
BATCH_LATENCY    = 1
BATCH_THROUGHPUT = 16
N_WARMUP         = 50
N_RUNS           = 200

# Initialize Model (ChangeFormerV6 is the finalized version in this repo)
# It takes embed_dim and output_nc as its primary arguments
model = ChangeFormerV6(embed_dim=256, output_nc=2).to(DEVICE).eval()

def benchmark_model(model, name="ChangeFormer"):
    t1 = torch.randn(BATCH_LATENCY, *INPUT_SHAPE).to(DEVICE)
    t2 = torch.randn(BATCH_LATENCY, *INPUT_SHAPE).to(DEVICE)

    print(f"\n{'='*60}")
    print(f"  {name} Benchmark — {torch.cuda.get_device_name(0)}")
    print(f"{'='*60}")

    # --- Params & GFLOPs ---
    model_cpu = copy.deepcopy(model).cpu()
    macs, params = profile(model_cpu, inputs=(t1.cpu(), t2.cpu()), verbose=False)
    gflops = macs * 2 / 1e9
    params_m = params / 1e6
    print(f"  Params  : {params_m:.2f} M")
    print(f"  GFLOPs  : {gflops:.2f}")

    # --- Latency ---
    with torch.no_grad():
        for _ in range(N_WARMUP):
            model(t1, t2)
    torch.cuda.synchronize()

    latencies = []
    with torch.no_grad():
        for _ in range(N_RUNS):
            torch.cuda.synchronize()
            start = time.perf_counter()
            model(t1, t2)
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - start) * 1000)
    
    lat_arr = np.array(latencies)
    lo, hi = np.percentile(lat_arr, [2, 98])
    trimmed = lat_arr[(lat_arr >= lo) & (lat_arr <= hi)]
    print(f"  Latency : {trimmed.mean():.2f} ± {trimmed.std():.2f} ms")

    # --- Throughput ---
    t1_b = torch.randn(BATCH_THROUGHPUT, *INPUT_SHAPE).to(DEVICE)
    t2_b = torch.randn(BATCH_THROUGHPUT, *INPUT_SHAPE).to(DEVICE)
    
    torch.cuda.synchronize()
    start_th = time.perf_counter()
    with torch.no_grad():
        for _ in range(100): 
            model(t1_b, t2_b)
    torch.cuda.synchronize()
    fps = (BATCH_THROUGHPUT * 100) / (time.perf_counter() - start_th)
    print(f"  Throughput: {fps:.1f} img/sec")

    # --- Peak Memory ---
    torch.cuda.reset_peak_memory_stats(DEVICE)
    with torch.no_grad():
        model(t1, t2)
    torch.cuda.synchronize()
    mem = torch.cuda.max_memory_allocated(DEVICE) / 1024**2
    print(f"  Peak Mem: {mem:.1f} MB")
    print(f"{'='*60}\n")

if __name__ == "__main__":
    benchmark_model(model, "ChangeFormer")

Cloning into 'ChangeFormer'...
remote: Enumerating objects: 1323, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 1323 (delta 73), reused 73 (delta 36), pack-reused 1199 (from 1)
Receiving objects: 100% (1323/1323), 14.04 MiB | 25.82 MiB/s, done.
Resolving deltas: 100% (761/761), done.


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)



  ChangeFormer Benchmark — Tesla T4
  Params  : 41.03 M
  GFLOPs  : 405.58
  Latency : 72.25 ± 1.18 ms
  Throughput: 14.3 img/sec
  Peak Mem: 483.9 MB



In [5]:
# 1. Clean and clone DTCDSCN
!rm -rf DTCDSCN
!git clone https://github.com/fitzpchao/DTCDSCN.git

# 2. Install dependencies
!pip install thop -q

# 3. Import and Benchmark
import sys
import os
import time
import numpy as np
import copy
import inspect

# Force PyTorch to load completely first
import torch
import torch.nn as nn
from thop import profile

# --- THE FIX ---
# The files are nested inside a SECOND folder named DTCDSCN.
REPO_PATH = '/kaggle/working/DTCDSCN/DTCDSCN'
sys.path.insert(0, REPO_PATH)

# Import the actual file found by your script
import model_sexp6

# Smartly grab the PyTorch model class from inside the file
try:
    DTCDSCN_Model = model_sexp6.CDNet34
    print("Successfully imported DTCDSCN (CDNet34).")
except AttributeError:
    try:
        DTCDSCN_Model = model_sexp6.CDNet
        print("Successfully imported DTCDSCN (CDNet).")
    except AttributeError:
        # If the author named it something totally different, 
        # dynamically find the main PyTorch Module inside the file
        classes = [obj for name, obj in inspect.getmembers(model_sexp6, inspect.isclass) 
                   if issubclass(obj, nn.Module) and obj.__module__ == 'model_sexp6']
        DTCDSCN_Model = classes[-1] # Usually the main network is the last/largest defined class
        print(f"Dynamically imported model class: {DTCDSCN_Model.__name__}")

# --- Config ---
DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_SHAPE      = (3, 256, 256)
BATCH_LATENCY    = 1
BATCH_THROUGHPUT = 16
N_WARMUP         = 50
N_RUNS           = 200

# Initialize Model
# DTCDSCN typically outputs multiple items due to deep supervision (DS).
class DTCDSCNWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x1, x2):
        # DTCDSCN often returns a list/tuple of outputs [main_out, ds1, ds2...]
        out = self.model(x1, x2)
        return out[0] if isinstance(out, (list, tuple)) else out

# Initialize with standard Siamese config
try:
    raw_model = DTCDSCN_Model(in_channels=3, num_classes=2).to(DEVICE).eval()
except TypeError:
    # Fallback if the author used different variable names for channels/classes
    raw_model = DTCDSCN_Model().to(DEVICE).eval()
    
model = DTCDSCNWrapper(raw_model)

# --- Standalone Benchmark Function ---
def benchmark_model(model, name="Model"):
    t1 = torch.randn(BATCH_LATENCY, *INPUT_SHAPE).to(DEVICE)
    t2 = torch.randn(BATCH_LATENCY, *INPUT_SHAPE).to(DEVICE)

    print(f"\n{'='*60}")
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"  {name} Benchmark — {device_name}")
    print(f"{'='*60}")

    # --- Params & GFLOPs ---
    model_cpu = copy.deepcopy(model).cpu()
    macs, params = profile(model_cpu, inputs=(t1.cpu(), t2.cpu()), verbose=False)
    gflops = macs * 2 / 1e9
    params_m = params / 1e6
    print(f"  Params  : {params_m:.2f} M")
    print(f"  GFLOPs  : {gflops:.2f}")

    # --- Latency ---
    with torch.no_grad():
        for _ in range(N_WARMUP):
            model(t1, t2)
    if torch.cuda.is_available(): torch.cuda.synchronize()

    latencies = []
    with torch.no_grad():
        for _ in range(N_RUNS):
            if torch.cuda.is_available(): torch.cuda.synchronize()
            start = time.perf_counter()
            model(t1, t2)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            latencies.append((time.perf_counter() - start) * 1000)
    
    lat_arr = np.array(latencies)
    lo, hi = np.percentile(lat_arr, [2, 98])
    trimmed = lat_arr[(lat_arr >= lo) & (lat_arr <= hi)]
    print(f"  Latency : {trimmed.mean():.2f} ± {trimmed.std():.2f} ms")

    # --- Throughput ---
    t1_b = torch.randn(BATCH_THROUGHPUT, *INPUT_SHAPE).to(DEVICE)
    t2_b = torch.randn(BATCH_THROUGHPUT, *INPUT_SHAPE).to(DEVICE)
    
    if torch.cuda.is_available(): torch.cuda.synchronize()
    start_th = time.perf_counter()
    with torch.no_grad():
        for _ in range(100): 
            model(t1_b, t2_b)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    fps = (BATCH_THROUGHPUT * 100) / (time.perf_counter() - start_th)
    print(f"  Throughput: {fps:.1f} img/sec")

    # --- Peak Memory ---
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)
        with torch.no_grad():
            model(t1, t2)
        torch.cuda.synchronize()
        mem = torch.cuda.max_memory_allocated(DEVICE) / 1024**2
        print(f"  Peak Mem: {mem:.1f} MB")
    else:
        print("  Peak Mem: N/A (Running on CPU)")
        
    print(f"{'='*60}\n")

if __name__ == "__main__":
    benchmark_model(model, "DTCDSCN")

Cloning into 'DTCDSCN'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 54 (delta 8), reused 23 (delta 8), pack-reused 30 (from 1)
Receiving objects: 100% (54/54), 68.86 KiB | 2.02 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Successfully imported DTCDSCN (CDNet34).

  DTCDSCN Benchmark — Tesla T4


/kaggle/working/DTCDSCN/DTCDSCN/model_sexp6.py:152: UserWarning: This overload of add is deprecated:
	add(Tensor input, Number alpha, Tensor other, *, Tensor out = None)
Consider using one of the following signatures instead:
	add(Tensor input, Tensor other, *, Number alpha = 1, Tensor out = None) (Triggered internally at /pytorch/torch/csrc/utils/python_arg_parser.cpp:1862.)
  return torch.add(chn_se, 1, spa_se)


  Params  : 41.07 M
  GFLOPs  : 40.78
  Latency : 25.21 ± 1.01 ms
  Throughput: 102.3 img/sec
  Peak Mem: 235.9 MB



In [4]:
!find /kaggle/working/DTCDSCN -name "*.py"

/kaggle/working/DTCDSCN/DTCDSCN/read_sexp6.py
/kaggle/working/DTCDSCN/DTCDSCN/utils_sexp6.py
/kaggle/working/DTCDSCN/DTCDSCN/model_sexp6.py
/kaggle/working/DTCDSCN/DTCDSCN/train_sexp6.py
/kaggle/working/DTCDSCN/DTCDSCN/module_part.py
/kaggle/working/DTCDSCN/DTCDSCN/cdloss.py
